# 💎 Aurigen — AI-Powered Jewelry Design Studio

> *Generate stunning, photorealistic jewelry designs using SDXL 1.0 + ControlNet with fine-tuned UNet weights.*

---

## ⚙️ Environment Requirements

| Setting | Value |
|---|---|
| **Accelerator** | GPU P100 ✅ |
| **Internet** | Enabled ✅ |
| **Runtime** | Python 3.12 |

> ⚠️ **Before running:** Make sure GPU is enabled in *Session options → Accelerator → GPU P100* and Internet is turned on.

---

## 🗂️ Notebook Overview

| Step | Description |
|---|---|
| **Cell 1** | Clone the repo from GitHub |
| **Cell 2** | Install required dependencies |
| **Cell 3** | Download fine-tuned UNet weights from Google Drive |
| **Cell 4** | Configure ngrok tunnel |
| **Cell 5** | Fix weights directory structure |
| **Cell 6** | Launch Streamlit app + expose via ngrok |
| **Cell 7** | Health check — verify Streamlit is running |

## Step 1 — Clone Repository

Clones the `main` branch which includes:
- fp16 precision loading
- `enable_model_cpu_offload()` to prevent CUDA OOM
- VAE slicing + tiling for memory efficiency
- DPMSolver scheduler for faster inference

In [ ]:
# Cell 1 — Clone repo
!git clone https://github.com/CodeNinjaSarthak/Aurigen-AI-Powered-Jewelry-Design-Studio.git
%cd Aurigen-AI-Powered-Jewelry-Design-Studio

## Step 2 — Install Dependencies

Installs packages not pre-bundled in the Kaggle Docker image.

| Package | Purpose |
|---|---|
| `diffusers` | SDXL + ControlNet pipeline |
| `transformers` | Text encoder / tokenizer |
| `accelerate` | CPU offloading support |
| `safetensors` | Fast model weight loading |
| `streamlit` | Web UI framework |

> ⏱️ Takes ~30 seconds.

In [ ]:
# Cell 2 — Install only what Kaggle/colab doesn't have
!pip install diffusers transformers accelerate safetensors streamlit -q

## Step 3 — Download Fine-Tuned Weights

Downloads the custom UNet checkpoint (`unet_epoch_3.pth` — ~5.1 GB) from Google Drive into the `fine-tuned-weights/` directory.

> ⏱️ Takes ~1–2 minutes depending on Drive speed.

In [ ]:
# Cell 3 — Download fine-tuned weights from Google Drive
!pip install gdown -q
!gdown --folder https://drive.google.com/drive/folders/13bx0xMu9Py2vFqFG8ocny2YVamw7EQOX \
    -O fine-tuned-weights/

# Verify weights downloaded
!ls fine-tuned-weights/

## Step 4 — Configure ngrok

Sets up the ngrok tunnel to expose the local Streamlit server publicly.

**Prerequisites:**
1. Create a free account at [ngrok.com](https://ngrok.com)
2. Copy your auth token from the ngrok dashboard
3. In Kaggle: go to **Add-ons → Secrets** and add a secret named `NGROK_SECRET_KEY` with your token

In [ ]:
# Cell 4 — Setup ngrok using Kaggle secret
!pip install pyngrok -q
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import requests

token = UserSecretsClient().get_secret("NGROK_SECRET_KEY")
ngrok.set_auth_token(token)

## Step 5 — Fix Weights Directory Structure

`gdown --folder` creates a nested subdirectory. This cell flattens it:

```
Before:  fine-tuned-weights/fine-tuned-weights/unet_epoch_3.pth
After:   fine-tuned-weights/unet_epoch_3.pth  ✅
```

Expected output: `README.md  unet_epoch_3.pth`

In [ ]:
!mv /kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio/fine-tuned-weights/fine-tuned-weights/unet_epoch_3.pth \
    /kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio/fine-tuned-weights/unet_epoch_3.pth

!rm -rf /kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio/fine-tuned-weights/fine-tuned-weights

# Verify
!ls /kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio/fine-tuned-weights/
# Should show: unet_epoch_3.pth  README.md

## Step 6 — Launch App

Starts the Streamlit server and exposes it via ngrok.

**What happens under the hood:**
- Streamlit runs on `localhost:8501`
- ngrok creates a public HTTPS tunnel to that port
- Model loading happens on first visit to the URL (~2–3 min for SDXL + ControlNet)

> ⏱️ Wait ~6 seconds after running for the public URL to appear.

> 💡 **Tip:** Keep this cell running — closing it kills the server.

In [ ]:
# Cell 5 — Launch Streamlit + expose via ngrok (with live output)
import subprocess, time, os, threading

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio"

proc = subprocess.Popen(
    ["streamlit", "run", "app/controlnet_app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    cwd="/kaggle/working/Aurigen-AI-Powered-Jewelry-Design-Studio",
    env=env
)

# Stream output live in background thread
def stream_output(p):
    for line in iter(p.stdout.readline, b""):
        print(line.decode("utf-8"), end="")

thread = threading.Thread(target=stream_output, args=(proc,), daemon=True)
thread.start()

time.sleep(6)
tunnel = ngrok.connect(8501)
print("=" * 55)
print(f"  ✨ Aurigen is live at: {tunnel.public_url}")
print("=" * 55)

## Step 7 — Health Check

Verifies the Streamlit process is still alive.

- ✅ **If output shows a PID** — app is running normally
- ❌ **If output is empty** — Streamlit crashed, re-run Step 6

Common crash reasons: CUDA OOM, missing weights file, import error.

In [ ]:
# Check streamlit process is still alive
import subprocess
result = subprocess.run(["pgrep", "-a", "streamlit"], capture_output=True, text=True)
print(result.stdout)
# If empty — streamlit crashed, need to rerun Cell 5

---

## 🛠️ Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Restart session, re-run all cells |
| `Pipeline failed to load` | Click **🔄 Retry Loading Model** in the app sidebar |
| `ngrok tunnel not found` | Check `NGROK_SECRET_KEY` secret is set correctly |
| `unet_epoch_3.pth not found` | Re-run Step 3 and Step 5 |
| Streamlit process empty in health check | Re-run Step 6 |

---

> Built with ❤️ by [CodeNinjaSarthak](https://github.com/CodeNinjaSarthak)